**Imports and shared constants**

Load the training libraries and set the two class names used in this run, Medium and Large, plus ImageNet normalization. The repository root is resolved so environment variables can be read from the project env file.

In [1]:
import json
from io import BytesIO
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
import boto3
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

CLASS_NAMES = ["Medium", "Large"]
NORMALIZATION = {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]}


from dotenv import load_dotenv
from IPython.display import display

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
load_dotenv(PROJECT_ROOT / ".env")


True

**Training settings and S3 connection**

Set the epoch count, batch size, learning rate, and S3 locations, then require a CUDA GPU. A unique output prefix keeps this run's artifacts separate until they are uploaded.

In [2]:
#local training settings and s3 paths
EPOCHS = 100
BATCH_SIZE = 16
LEARNING_RATE = 0.001
AWS_PROFILE = os.environ.get("AWS_PROFILE", "ATLAS")
AWS_REGION = "us-east-2"
BUCKET = "atlas-acm-research-f26"
IMAGE_PREFIX = "satellite/processed/"
METADATA_KEY = "dataset/im3_atlas_cleaned.csv"
OUTPUT_PREFIX = "artifacts/baseline/"

#all data and results stay in memory until results are uploaded to S3.
job_name = datetime.now(timezone.utc).strftime("local-%Y%m%d-%H%M%S-%f")
s3_output = f"{OUTPUT_PREFIX.rstrip('/')}/{job_name}/"

torch.manual_seed(42)
print(f"Python: {sys.executable}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required. Select the atlas .venv kernel with CUDA-enabled PyTorch."
    )
device = torch.device("cuda")
print(f"PyTorch: {torch.__version__} | CUDA build: {torch.version.cuda}")
print(f"Training GPU: {torch.cuda.get_device_name(0)}")
    
available_profiles = boto3.Session().available_profiles
aws_session = boto3.Session(
    profile_name=AWS_PROFILE if AWS_PROFILE in available_profiles else None,
    region_name=AWS_REGION,
)
if aws_session.get_credentials() is None:
    raise RuntimeError(
        "AWS credentials are missing. Configure the ATLAS profile with "
        "aws login --profile ATLAS --region us-east-2" 
    )
s3 = aws_session.client("s3")
print(f"Run: {job_name} | Device: {device}")


Python: c:\Users\kevin\Projects\ATLAS\.venv\Scripts\python.exe
PyTorch: 2.11.0+cu128 | CUDA build: 12.8
Training GPU: NVIDIA GeForce RTX 5070
Run: local-20261105-032521-267309 | Device: cuda


**Load labels and list image tensors**

Read the facility table from S3, drop the unused small class, and remap the remaining labels to two classes. Listing the tensor keys up front lets later cells cache each image in memory instead of downloading it every epoch.

In [5]:
#read CSV into RAM and list tensor keys
def read_s3_bytes(key):
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"]
    try:
        return body.read()
    finally:
        body.close()

metadata = pd.read_csv(
    BytesIO(read_s3_bytes(METADATA_KEY)), dtype={"id": str}
)
metadata = metadata.loc[metadata["size_class"] != 0].copy()
metadata["size_class"] = metadata["size_class"].map({1: 0, 2: 1})
tensor_keys = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=IMAGE_PREFIX):
    tensor_keys.extend(item["Key"] for item in page.get("Contents", []) if item["Key"].endswith(".pt"))
if not tensor_keys:
    raise ValueError(f"No tensors found at s3://{BUCKET}/{IMAGE_PREFIX}")
tensor_cache = {}  #shared CPU RAM cache avoids fetching the same tensor each epoch.
print(f"Found {len(tensor_keys)} tensors in S3; metadata loaded into memory.")


Found 180 tensors in S3; metadata loaded into memory.


**Dataset, split, and loaders**

Define a dataset that reads cached image tensors, optionally flips training images, and applies ImageNet normalization. Facilities are then split into stratified train, validation, and test sets with matching data loaders.

In [6]:
# Baseline dataset reads tensors from S3 into RAM.
class DataCenterDataset(Dataset):
    def __init__(self, frame, augment=False):
        self.rows = list(
            frame[["s3_key", "size_class"]].itertuples(
                index=False, name=None
            )
        )

        self.augment = augment
        self.flips = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
        ])
        self.normalize = transforms.Normalize(**NORMALIZATION)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        key, label = self.rows[index]

        if key not in tensor_cache:
            data = torch.load(
                BytesIO(read_s3_bytes(key)),
                map_location="cpu",
                weights_only=True,
            )
            tensor_cache[key] = data["image"].float()

        image = tensor_cache[key].clone()

        if self.augment:
            k = torch.randint(0, 4, ()).item()
            image = torch.rot90(image, k, dims=(-2, -1))
            image = self.flips(image)

        return self.normalize(image), int(label)


def normalize_id(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    return str(int(value)) if value.isdecimal() else value


def load_and_split_data(tensor_keys, metadata):
    metadata = metadata.copy()
    metadata["id"] = metadata["id"].map(normalize_id)

    # Use the target already provided by im3_atlas_cleaned.csv.
    if "size_class" not in metadata.columns:
        raise ValueError("Metadata is missing size_class.")

    labels = pd.to_numeric(metadata["size_class"], errors="coerce")

    if not labels.isin([0, 1]).all():
        raise ValueError(
            "Expected size_class: 0=Medium, 1=Large, with no missing values."
        )

    metadata["size_class"] = labels.astype(int)

    records = [
        {
            "id": normalize_id(Path(key).stem),
            "s3_key": key,
        }
        for key in tensor_keys
    ]

    images = pd.DataFrame(records).merge(
        metadata[["id", "size_class"]],
        on="id",
        validate="many_to_one",
    )

    print(
        f"Matched {len(images)} tensors "
        f"across {images['id'].nunique()} facilities"
    )

    facilities = images[["id", "size_class"]].drop_duplicates()

    train, remainder = train_test_split(
        facilities,
        test_size=0.30,
        random_state=None,
        stratify=facilities["size_class"],
    )

    validation, test = train_test_split(
        remainder,
        test_size=0.50,
        random_state=42,
        stratify=remainder["size_class"],
    )

    return [
        images[images["id"].isin(split["id"])].copy()
        for split in (train, validation, test)
    ]


frames = load_and_split_data(tensor_keys, metadata)

loaders = [
    DataLoader(
        DataCenterDataset(frame, augment=(index == 0)),
        batch_size=BATCH_SIZE,
        shuffle=(index == 0),
        num_workers=0,
    )
    for index, frame in enumerate(frames)
]

for name, frame in zip(("train", "validation", "test"), frames):
    print(
        f"{name}: {len(frame)} tensors, "
        f"{frame['id'].nunique()} facilities"
    )

print("size_class: 0=Medium, 1=Large")

Matched 173 tensors across 173 facilities
train: 121 tensors, 121 facilities
validation: 26 tensors, 26 facilities
test: 26 tensors, 26 facilities
size_class: 0=Medium, 1=Large


**Train the classification head**

Freeze a pretrained ResNet-18 and replace its final layer with a two-class head. Each epoch updates only that head and records train and validation loss and accuracy.

In [7]:
def run_epoch(model, loader, criterion, device, optimizer=None):
    model.eval()
    if optimizer is not None:
        model.fc.train()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(optimizer is not None):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            #clear the previous batch's gradients before calculating new ones.
            if optimizer is not None:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            #calculate gradients and use them to update the trainable weights.
            if optimizer is not None:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

#reuse pretrained image features and train a new three-class classification head.
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for parameter in model.parameters():
    parameter.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
model.to(device)

#cross-entropy measures classification error; Adam updates only the new head.
class_counts = (
    frames[0]["size_class"]
    .value_counts()
    .reindex(range(len(CLASS_NAMES)), fill_value=0)
)

class_weights = torch.tensor(
    len(frames[0]) / (len(CLASS_NAMES) * class_counts.to_numpy()),
    dtype=torch.float32,
    device=device,
)

criterion = nn.CrossEntropyLoss(weight=class_weights)
print("Class weights:", class_weights.cpu().tolist())
    
optimizer = torch.optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)

history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(
        model, loaders[0], criterion, device, optimizer
    )
    val_loss, val_accuracy = run_epoch(
        model, loaders[1], criterion, device
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
    })

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.2%} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.2%}",
        flush=True,
    )


Class weights: [1.7285714149475098, 0.7034883499145508]
Epoch 1/100 | Train Loss: 0.7589 | Train Acc: 52.07% | Val Loss: 0.7019 | Val Acc: 38.46%
Epoch 2/100 | Train Loss: 0.6889 | Train Acc: 47.93% | Val Loss: 0.6386 | Val Acc: 80.77%
Epoch 3/100 | Train Loss: 0.6484 | Train Acc: 73.55% | Val Loss: 0.5996 | Val Acc: 73.08%
Epoch 4/100 | Train Loss: 0.6015 | Train Acc: 59.50% | Val Loss: 0.5732 | Val Acc: 76.92%
Epoch 5/100 | Train Loss: 0.5858 | Train Acc: 76.03% | Val Loss: 0.5608 | Val Acc: 84.62%
Epoch 6/100 | Train Loss: 0.5111 | Train Acc: 76.86% | Val Loss: 0.5221 | Val Acc: 76.92%
Epoch 7/100 | Train Loss: 0.5046 | Train Acc: 72.73% | Val Loss: 0.5001 | Val Acc: 84.62%
Epoch 8/100 | Train Loss: 0.5803 | Train Acc: 75.21% | Val Loss: 0.5199 | Val Acc: 80.77%
Epoch 9/100 | Train Loss: 0.5157 | Train Acc: 72.73% | Val Loss: 0.4743 | Val Acc: 76.92%
Epoch 10/100 | Train Loss: 0.4586 | Train Acc: 79.34% | Val Loss: 0.5083 | Val Acc: 80.77%
Epoch 11/100 | Train Loss: 0.4584 | Train A

**Score the held-out test set**

Run the trained model on the test loader and store accuracy, macro-F1, the classification report, and a confusion-matrix figure in memory. Nothing is uploaded until the following save step.

In [8]:
#produce metrics and plot in RAM
def figure_bytes(figure):
    buffer = BytesIO()
    figure.savefig(buffer, format="png", dpi=150)
    return buffer.getvalue()

def save_evaluation(model, loader, device, history, results):
    predictions, actual = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in loader:
            predictions.extend(model(images.to(device)).argmax(1).cpu().tolist())
            actual.extend(labels.tolist())
    #rows are actual classes, columns are predicted classes.
    matrix = confusion_matrix(actual, predictions, labels=[0, 1])
    report_args = dict(labels=[0, 1], target_names=CLASS_NAMES, zero_division=0)
    report = classification_report(actual, predictions, **report_args)
    metrics = {
        "test_accuracy": float(accuracy_score(actual, predictions)),
        "test_macro_f1": float(
            f1_score(
                actual,
                predictions,
                labels=[0, 1],
                average="macro",
                zero_division=0,
            )
        ),
        "classification_report": classification_report(actual, predictions, output_dict=True, **report_args),
        "confusion_matrix": matrix.tolist(),
    }
    results["metrics.json"] = json.dumps(metrics, indent=2).encode("utf-8")
    results["classification_report.txt"] = report.encode("utf-8")
    results["training_history.csv"] = pd.DataFrame(history).to_csv(index=False).encode("utf-8")
    print(f"Test accuracy: {metrics['test_accuracy']:.4f} | Macro F1: {metrics['test_macro_f1']:.4f}")
    print(report, flush=True)

    #compare training and validation over time to show learning and possible overfitting.
    for metric in ("loss", "accuracy"):
        fig, ax = plt.subplots()
        for split, label in (("train", "Training"), ("val", "Validation")):
            ax.plot([row["epoch"] for row in history],
                    [row[f"{split}_{metric}"] for row in history], label=label)
        ax.set(xlabel="Epoch", ylabel=metric.capitalize(),
               title=f"Training and validation {metric}")
        if metric == "accuracy":
            ax.set_ylim(0, 1)
        ax.legend()
        fig.tight_layout()
        results[f"{metric}_curve.png"] = figure_bytes(fig)
        plt.close(fig)
    display = ConfusionMatrixDisplay(matrix, display_labels=CLASS_NAMES)
    display.plot(cmap="Blues", values_format="d")
    display.ax_.set_title("Test confusion matrix")
    display.figure_.tight_layout()
    results["confusion_matrix.png"] = figure_bytes(display.figure_)
    plt.close(display.figure_)

results = {}
save_evaluation(model, loaders[2], device, history, results)
results["data_splits.csv"] = pd.concat([
    frame.assign(split=name)
    for name, frame in zip(("train", "validation", "test"), frames)
]).to_csv(index=False).encode("utf-8")
#save trained weights in RAM, the upload cell sends this under the current job.
buffer = BytesIO()
torch.save({name: value.detach().cpu() for name, value in model.state_dict().items()}, buffer)
results["model.pkl"] = buffer.getvalue()
print("Results prepared in memory:", ", ".join(sorted(results)))


Test accuracy: 0.7692 | Macro F1: 0.7292
              precision    recall  f1-score   support

      Medium       0.62      0.62      0.62         8
       Large       0.83      0.83      0.83        18

    accuracy                           0.77        26
   macro avg       0.73      0.73      0.73        26
weighted avg       0.77      0.77      0.77        26

Results prepared in memory: accuracy_curve.png, classification_report.txt, confusion_matrix.png, data_splits.csv, loss_curve.png, metrics.json, model.pkl, training_history.csv


**Grad-CAM examples**

Build Grad-CAM maps from the last convolutional block for one correct and one incorrect test image in each class. This shows which parts of the satellite chip the head used for those predictions.

In [9]:
#grad-cam one correct and one incorrect example per actual class
import torch.nn.functional as F
from IPython.display import display

model.eval()
target_layer = model.layer4[-1]


def make_gradcam(image, target_class):
    captured = {}

    def capture_features(module, inputs, output):
        captured["features"] = output

    handle = target_layer.register_forward_hook(capture_features)

    try:
        with torch.enable_grad():
            #required because the backbone is frozen.
            x = image.unsqueeze(0).to(device).detach().requires_grad_(True)
            logits = model(x)
            features = captured["features"]

            gradients = torch.autograd.grad(
                logits[0, target_class], features
            )[0]

            weights = gradients.mean(dim=(2, 3), keepdim=True)
            cam = F.relu((weights * features).sum(dim=1, keepdim=True))
            cam = F.interpolate(
                cam,
                size=image.shape[-2:],
                mode="bilinear",
                align_corners=False,
            ).detach().squeeze(0).squeeze(0)

            maximum = cam.max()
            if maximum > 0:
                cam = cam / maximum
            else:
                print("No positive Grad-CAM signal for this sample/class.")

            return cam.cpu().numpy()
    finally:
        handle.remove()


test_dataset = loaders[2].dataset
if getattr(test_dataset, "augment", False):
    raise RuntimeError("Test images must not use random augmentation.")

categories = [
    (class_index, outcome)
    for class_index in range(len(CLASS_NAMES))
    for outcome in ("correct", "incorrect")
]
selected = {}

#select the first available example for each class/outcome combination.
with torch.no_grad():
    for index in range(len(test_dataset)):
        image, actual = test_dataset[index]
        actual = int(actual)

        probabilities = model(
            image.unsqueeze(0).to(device)
        ).softmax(dim=1)[0].cpu()

        predicted = int(probabilities.argmax().item())
        outcome = "correct" if predicted == actual else "incorrect"
        category = (actual, outcome)

        if category not in selected:
            row = frames[2].iloc[index]
            selected[category] = {
                "image": image,
                "id": str(row["id"]),
                "s3_key": row["s3_key"],
                "actual": actual,
                "predicted": predicted,
                "probabilities": probabilities.tolist(),
            }

        if len(selected) == len(categories):
            break

#replace only the previous Grad-CAM results held in RAM.
for name in list(results):
    if name.startswith("gradcam/"):
        del results[name]

sample_details = []
missing = []

for actual_class, outcome in categories:
    class_name = CLASS_NAMES[actual_class]
    category = (actual_class, outcome)

    if category not in selected:
        message = f"No {outcome} {class_name} prediction in the test set."
        print(message)
        missing.append({
            "actual_class": class_name,
            "outcome": outcome,
        })
        continue

    sample = selected[category]
    predicted = sample["predicted"]
    heatmap = make_gradcam(sample["image"], predicted)

    original = (
        tensor_cache[sample["s3_key"]]
        .detach()
        .cpu()
        .permute(1, 2, 0)
        .clamp(0, 1)
        .numpy()
    )

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(original)
    axes[0].set_title("Original satellite image")

    axes[1].imshow(original)
    overlay = axes[1].imshow(
        heatmap, cmap="jet", alpha=0.45, vmin=0, vmax=1
    )
    axes[1].set_title(
        f"Grad-CAM for predicted class: {CLASS_NAMES[predicted]}"
    )
    fig.colorbar(overlay, ax=axes[1], fraction=0.046, pad=0.04)

    for axis in axes:
        axis.axis("off")

    probability = sample["probabilities"][predicted]
    fig.suptitle(
        f"{outcome.upper()} | Facility {sample['id']}\n"
        f"Actual: {class_name} | "
        f"Predicted: {CLASS_NAMES[predicted]} ({probability:.1%})"
    )
    fig.tight_layout()

    filename = (
        f"gradcam/{class_name.lower()}_{outcome}_{sample['id']}.png"
    )
    results[filename] = figure_bytes(fig)

    display(fig)
    plt.close(fig)

    sample_details.append({
        "outcome": outcome,
        "facility_id": sample["id"],
        "s3_key": sample["s3_key"],
        "actual_class": class_name,
        "predicted_class": CLASS_NAMES[predicted],
        "class_probabilities": dict(
            zip(CLASS_NAMES, sample["probabilities"])
        ),
        "target_layer": "layer4[-1]",
        "image_file": filename,
    })

results["gradcam/samples.json"] = json.dumps(
    {
        "job_name": job_name,
        "grouped_by": "actual_class",
        "samples": sample_details,
        "missing_categories": missing,
    },
    indent=2,
).encode("utf-8")

print(f"Prepared {len(sample_details)} of {len(categories)} Grad-CAM examples.")

<Figure size 1000x500 with 3 Axes>

<Figure size 1000x500 with 3 Axes>

<Figure size 1000x500 with 3 Axes>

<Figure size 1000x500 with 3 Axes>

Prepared 4 of 4 Grad-CAM examples.


**Upload artifacts to S3**

Send the in-memory metrics, figures, and model file to this run's S3 prefix. The cell stops if the evaluation step has not prepared the model payload.

In [10]:
#upload to s3 from RAM
if "model.pkl" not in results:
    raise RuntimeError("Run the evaluation cell to prepare model.pkl before uploading.")
content_types = {".png": "image/png", ".json": "application/json",
                 ".csv": "text/csv", ".txt": "text/plain", ".pkl": "application/octet-stream"}
for name, payload in sorted(results.items()):
    s3.upload_fileobj(BytesIO(payload), BUCKET, f"{s3_output}{name}",
                      ExtraArgs={"ContentType": content_types[Path(name).suffix]})
    print(f"Uploaded: {name}")
print(f"Metrics and visuals: s3://{BUCKET}/{s3_output}")


Uploaded: accuracy_curve.png
Uploaded: classification_report.txt
Uploaded: confusion_matrix.png
Uploaded: data_splits.csv
Uploaded: gradcam/large_correct_25706343.png
Uploaded: gradcam/large_incorrect_744177375.png
Uploaded: gradcam/medium_correct_15884451.png
Uploaded: gradcam/medium_incorrect_225745958.png
Uploaded: gradcam/samples.json
Uploaded: loss_curve.png
Uploaded: metrics.json
Uploaded: model.pkl
Uploaded: training_history.csv
Metrics and visuals: s3://atlas-acm-research-f26/artifacts/baseline/local-20261105-032521-267309/
